In [1]:
from graphein.protein.config import ProteinGraphConfig
from graphein.protein.graphs import construct_graph
from graphein.protein.edges.distance import add_distance_threshold, add_peptide_bonds,add_k_nn_edges
from functools import partial
from graphein.protein.features.nodes.amino_acid import *
from graphein.protein.features.nodes.geometry import *
from graphein.protein.features.nodes import dssp
from graphein.protein.config import ProteinGraphConfig, DSSPConfig
from graphein.protein.features.nodes.dssp import *
import networkx as nx
from graphein.protein.visualisation import plotly_protein_structure_graph
import os
import glob
import matplotlib.pyplot as plt

/home/runchangjia/.conda/envs/graphin/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
params_to_change = {"granularity": "CA","edge_construction_functions": [add_peptide_bonds,partial(add_distance_threshold, long_interaction_threshold=5, threshold=5)],"node_metadata_functions": [meiler_embedding,expasy_protein_scale,hydrogen_bond_acceptor,hydrogen_bond_donor]}#,add_beta_carbon_vector,add_sequence_neighbour_vector,add_sidechain_vectoramino_acid_one_hot,asa,phi,psi,rsa
config = ProteinGraphConfig(**params_to_change)
config.dict()

/tmp/ipykernel_2990223/4062411386.py:3: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.10/migration/
  config.dict()


{'granularity': 'CA',
 'keep_hets': [],
 'insertions': True,
 'alt_locs': 'max_occupancy',
 'pdb_dir': None,
 'verbose': False,
 'exclude_waters': True,
 'deprotonate': False,
 'protein_df_processing_functions': None,
 'edge_construction_functions': [<function graphein.protein.edges.distance.add_peptide_bonds(G: 'nx.Graph') -> 'nx.Graph'>,
  functools.partial(<function add_distance_threshold at 0x7fd6b6045af0>, long_interaction_threshold=5, threshold=5)],
 'node_metadata_functions': [<function graphein.protein.features.nodes.amino_acid.meiler_embedding(n: str, d: Dict[str, Any], return_array: bool = False) -> Union[pandas.core.series.Series, numpy.ndarray]>,
  <function graphein.protein.features.nodes.amino_acid.expasy_protein_scale(n: str, d: Dict[str, Any], selection: Union[List[str], NoneType] = None, add_separate: bool = False, return_array: bool = False) -> Union[pandas.core.series.Series, numpy.ndarray]>,
  <function graphein.protein.features.nodes.amino_acid.hydrogen_bond_accept

In [ ]:
import os
import glob
from pathlib import Path
# protein pdb path with uniprot id.
path = r'/home/runchangjia/project2/benign_test'
import pickle
pdb_files = glob.glob(os.path.join(path, '*.pdb'))
graph_list = []
graph_dict ={}
id_list = []
for pdb_file in pdb_files:
    # print(pdb_file)
    graph_list.append(construct_graph(config=config, path=pdb_file))
    id_list.append(pdb_file[-10:-4])

In [ ]:
#add 3 features.
for g in graph_list:
    add_beta_carbon_vector(g)
    add_sidechain_vector(g)
    add_sequence_neighbour_vector(g)
    

In [5]:
for i,v in enumerate(id_list):
    graph_dict[v]= graph_list[i]

In [ ]:

import pickle
# /home/runchangjia/project2/benign_test.pkl
with open('/home/runchangjia/project2/uncertain/id_site.pkl', 'rb') as f:
    id_dict = pickle.load(f)

tmp=[]
cord =[]
cord_dict1 ={}#id
for i, g in enumerate(graph_list):
    tmp=[]
    for n, d in g.nodes(data=True):
        tmp.append((d["coords"][0], d["coords"][1], d["coords"][2]))
    cord_dict1[id_list[i]] = tmp


In [ ]:
cord = []
sub_list = []
sub_id_list = []
from graphein.protein.subgraphs import extract_subgraph
# extract sub graph with 8 A.
for i,(k,v) in enumerate(id_dict.items()):
    for s in v:
        print(type(k),type(s))
        site=cord_dict1[k][s-1]#x,y,z
        print(k,s,site)#232个子图
        sub_id_list.append(k+'_'+str(s))
        sub_list.append(extract_subgraph(graph_dict[k], centre_point=site, radius=8))
       
        

<class 'str'> <class 'int'>
O00187 128 (-61.657, 5.543, 86.883)
<class 'str'> <class 'int'>
O00217 102 (-9.76, -1.205, 14.565)
<class 'str'> <class 'int'>
O00754 99 (-18.177, 22.52, 11.073)
<class 'str'> <class 'int'>
O00754 1000 (16.122, 5.896, 16.648)
<class 'str'> <class 'int'>
O00754 956 (8.331, 2.568, 14.773)
<class 'str'> <class 'int'>
O00754 950 (23.175, 10.629, 21.868)
<class 'str'> <class 'int'>
O00754 916 (15.321, 3.77, 10.729)
<class 'str'> <class 'int'>
O00754 892 (28.365, 6.885, 4.853)
<class 'str'> <class 'int'>
O00754 891 (27.155, 10.238, 3.415)
<class 'str'> <class 'int'>
O00754 801 (-1.521, -9.616, -10.011)
<class 'str'> <class 'int'>
O00754 745 (2.871, -11.672, -3.628)
<class 'str'> <class 'int'>
O00754 565 (23.755, 6.013, -22.376)
<class 'str'> <class 'int'>
O00754 501 (15.126, 17.106, -20.726)
<class 'str'> <class 'int'>
O00754 457 (-9.785, 14.997, -6.928)
<class 'str'> <class 'int'>
O00754 453 (-11.599, 9.23, -6.132)
<class 'str'> <class 'int'>
O00754 445 (-5.532, 

In [10]:
for i,s_g in enumerate(sub_list):

    node_dict={}
    for nodename in list(s_g.nodes):
        node_dict[str(nodename.split(':')[2])]=nodename

    tmp = sub_id_list[i].split('_')[1]
    print(sub_id_list[i],tmp)
    center = node_dict[tmp]

    for node in s_g.nodes:
        if node == center:
            continue
        # if not s_g.has_edge(node, center):
        s_g.add_edge(node, center,kind={'distance_threshold'})

O00187_128 128
O00217_102 102
O00754_99 99
O00754_1000 1000
O00754_956 956
O00754_950 950
O00754_916 916
O00754_892 892
O00754_891 891
O00754_801 801
O00754_745 745
O00754_565 565
O00754_501 501
O00754_457 457
O00754_453 453
O00754_445 445
O00754_420 420
O00754_402 402
O00754_390 390
O00754_379 379
O00754_355 355
O00754_318 318
O00754_202 202
O00754_200 200
O00754_159 159
O00754_95 95
O00754_74 74
O00754_55 55
O14521_145 145
O14618_163 163
O14745_110 110
O14745_153 153
O14746_694 694
O14746_1090 1090
O14746_811 811
O14746_901 901
O14746_791 791
O14746_867 867
O14746_170 170
O14746_726 726
O14746_979 979
O14746_1110 1110
O14773_266 266
O14773_175 175
O14802_558 558
O15305_229 229
O43520_203 203
O43520_500 500
O43766_249 249
O43918_15 15
O60260_56 56
O60566_844 844
O60566_921 921
O60566_814 814
O75398_264 264
O75746_353 353
O75792_291 291
O75923_948 948
O75923_85 85
O95180_282 282
O95180_831 831
O95243_562 562
O95450_331 331
O95932_517 517
O96017_181 181
O96017_239 239
O96017_371 371
O96

In [11]:
# NBVAL_SKIP
from graphein.ml.conversion import GraphFormatConvertor

format_convertor = GraphFormatConvertor('nx', 'pyg',
                                        verbose = 'gnn',
                                        columns = None)

In [12]:
from tqdm import tqdm
from torch_geometric.data import Data
pyg_list = [format_convertor(graph) for graph in sub_list]

In [13]:
len(pyg_list)

611

In [14]:
for k,v in enumerate(pyg_list):
    pyg_list[k].id = sub_id_list[k]

In [15]:
pyg_list

[Data(edge_index=[2, 44], node_id=[14], coords=[14, 3], name='O00187', num_nodes=14, id='O00187_128'),
 Data(edge_index=[2, 46], node_id=[16], coords=[16, 3], name='O00217', num_nodes=16, id='O00217_102'),
 Data(edge_index=[2, 32], node_id=[11], coords=[11, 3], name='O00754', num_nodes=11, id='O00754_99'),
 Data(edge_index=[2, 38], node_id=[13], coords=[13, 3], name='O00754', num_nodes=13, id='O00754_1000'),
 Data(edge_index=[2, 36], node_id=[13], coords=[13, 3], name='O00754', num_nodes=13, id='O00754_956'),
 Data(edge_index=[2, 22], node_id=[8], coords=[8, 3], name='O00754', num_nodes=8, id='O00754_950'),
 Data(edge_index=[2, 42], node_id=[13], coords=[13, 3], name='O00754', num_nodes=13, id='O00754_916'),
 Data(edge_index=[2, 38], node_id=[13], coords=[13, 3], name='O00754', num_nodes=13, id='O00754_892'),
 Data(edge_index=[2, 34], node_id=[12], coords=[12, 3], name='O00754', num_nodes=12, id='O00754_891'),
 Data(edge_index=[2, 42], node_id=[14], coords=[14, 3], name='O00754', num_n

In [ ]:
import pandas as pd
import torch
import numpy as np
c_beta_vectorL,sidechain_vectorL,sequence_neighbour_vector_n_to_cL=[],[],[]
coordL,bfactorL=[],[]
meilerDF=pd.DataFrame()
expasyDF=pd.DataFrame()
dft2=pd.DataFrame()
dft3=pd.DataFrame()
#add  node meiler expasy features.
for i,g in enumerate(sub_list):
    dft2=pd.DataFrame()
    dft3=pd.DataFrame()
    expasyDF=pd.DataFrame()
    meilerDF=pd.DataFrame()
    for n, d in g.nodes(data=True):
        df1 = d["expasy"].to_frame()
        df2 = pd.DataFrame(df1.values.T) 
        expasyDF=pd.concat([expasyDF,df2], axis=0)
    expasyDF.reset_index(drop=True, inplace=True)
    # print(expasyDF.shape)61
    
    for n, d in g.nodes(data=True):
        df3 = d["meiler"].to_frame()
        df4 = pd.DataFrame(df3.values.T) 
        meilerDF=pd.concat([meilerDF,df4], axis=0)
    meilerDF.reset_index(drop=True, inplace=True)
    # print(meilerDF.shape)7

    c_beta_vectorL,sidechain_vectorL,sequence_neighbour_vector_n_to_cL=[],[],[]
    coordL,bfactorL=[],[]
    coordFeaDF=pd.DataFrame()
    bfactorDF=pd.DataFrame()
    c_beta_vectorFeaDF =pd.DataFrame()
    sidechain_vectorFeaDF = pd.DataFrame()
    sequence_neighbour_vector_n_to_cFeaDF = pd.DataFrame()
    
    for n, d in g.nodes(data=True):
        c_beta_vectorL.append(d['c_beta_vector'])
        sidechain_vectorL.append(d['sidechain_vector'])
        sequence_neighbour_vector_n_to_cL.append(d['sequence_neighbour_vector_n_to_c'])
        bfactorL.append(d['b_factor'])
        coordL.append(d['coords'])

    coordFeaDF = pd.DataFrame(np.array(coordL),columns=['coord_'+str(i) for i in range(len(coordL[0]))])
    print(coordFeaDF.shape)
    bfactorDF = pd.DataFrame(np.array(bfactorL),columns=['bfacor'])
    print(bfactorDF.shape)
    c_beta_vectorFeaDF=pd.DataFrame(np.array(c_beta_vectorL),columns=['c_beta_vector_'+str(i) for i in range(len(c_beta_vectorL[0]))])
    print(c_beta_vectorFeaDF.shape)
    sidechain_vectorFeaDF=pd.DataFrame(np.array(sidechain_vectorL),columns=['sidechain_vector_'+str(i) for i in range(len(sidechain_vectorL[0]))])
    print(sidechain_vectorFeaDF.shape)
    sequence_neighbour_vector_n_to_cFeaDF=pd.DataFrame(np.array(sequence_neighbour_vector_n_to_cL),columns=['sequence_neighbour_vector_n_to_c_'+str(i) for i in range(len(sequence_neighbour_vector_n_to_cL[0]))])
    print(sequence_neighbour_vector_n_to_cFeaDF.shape)
    dft2 = pd.concat([coordFeaDF,bfactorDF,c_beta_vectorFeaDF,sidechain_vectorFeaDF,sequence_neighbour_vector_n_to_cFeaDF],axis=1)
    dft3 = pd.concat([expasyDF, meilerDF ,dft2],axis=1)
    # print(dft3.shape)
    pyg_list[i].x=torch.FloatTensor(dft3.values.astype(float))
    pyg_list[i].y=torch.LongTensor([0])#long tensor


(14, 3)
(14, 1)
(14, 3)
(14, 3)
(14, 3)
(16, 3)
(16, 1)
(16, 3)
(16, 3)
(16, 3)
(11, 3)
(11, 1)
(11, 3)
(11, 3)
(11, 3)
(13, 3)
(13, 1)
(13, 3)
(13, 3)
(13, 3)
(13, 3)
(13, 1)
(13, 3)
(13, 3)
(13, 3)
(8, 3)
(8, 1)
(8, 3)
(8, 3)
(8, 3)
(13, 3)
(13, 1)
(13, 3)
(13, 3)
(13, 3)
(13, 3)
(13, 1)
(13, 3)
(13, 3)
(13, 3)
(12, 3)
(12, 1)
(12, 3)
(12, 3)
(12, 3)
(14, 3)
(14, 1)
(14, 3)
(14, 3)
(14, 3)
(15, 3)
(15, 1)
(15, 3)
(15, 3)
(15, 3)
(14, 3)
(14, 1)
(14, 3)
(14, 3)
(14, 3)
(13, 3)
(13, 1)
(13, 3)
(13, 3)
(13, 3)
(14, 3)
(14, 1)
(14, 3)
(14, 3)
(14, 3)
(13, 3)
(13, 1)
(13, 3)
(13, 3)
(13, 3)
(17, 3)
(17, 1)
(17, 3)
(17, 3)
(17, 3)
(13, 3)
(13, 1)
(13, 3)
(13, 3)
(13, 3)
(12, 3)
(12, 1)
(12, 3)
(12, 3)
(12, 3)
(15, 3)
(15, 1)
(15, 3)
(15, 3)
(15, 3)
(14, 3)
(14, 1)
(14, 3)
(14, 3)
(14, 3)
(15, 3)
(15, 1)
(15, 3)
(15, 3)
(15, 3)
(12, 3)
(12, 1)
(12, 3)
(12, 3)
(12, 3)
(14, 3)
(14, 1)
(14, 3)
(14, 3)
(14, 3)
(14, 3)
(14, 1)
(14, 3)
(14, 3)
(14, 3)
(15, 3)
(15, 1)
(15, 3)
(15, 3)
(15, 3)
(12, 

In [17]:
len(pyg_list)

118

In [18]:
pyg_list[16].node_id

['A:ARG:1932',
 'A:ILE:1933',
 'A:CYS:1935',
 'A:PHE:1907',
 'A:SER:1906',
 'A:HIS:1934',
 'A:LEU:1937',
 'A:GLY:1936']

In [17]:
import torch
import pickle
import numpy as np
def _positional_embeddings(edge_index, num_embeddings=6, period_range=[2, 1000]):
        
        d = edge_index[0] - edge_index[1]
     
        frequency = torch.exp(
            torch.arange(0, num_embeddings, 2, dtype=torch.float32)
            * -(np.log(10000.0) / num_embeddings)
        )
        angles = d.unsqueeze(-1) * frequency
        E = torch.cat((torch.cos(angles), torch.sin(angles)), -1)
        return E

In [ ]:
import torch
import numpy as np
# add edge features.
def add_edge_features(data):
   
    coords = data.coords  #shape is (num_nodes, 3)
    edge_index = data.edge_index  # edge index,the shape is (2, N).
    
    # 计算每条边的欧式距离
    edge_start_coords = coords[edge_index[0]]  # start coords
    edge_end_coords = coords[edge_index[1]]  # end coords
    
    # 计算两点之间的欧氏距离
    distance = torch.sqrt(torch.sum((edge_start_coords - edge_end_coords) ** 2, dim=1))
    
    # 将欧氏距离作为边特征
    edge_feature = distance.unsqueeze(1)  # 转换为形状为 (num_edges, 1) 的列向量

    E = _positional_embeddings(edge_index)
    # 将edge_feature添加到data对象中
    edge_attr = torch.cat([edge_feature, E], dim=-1) 

    data.edge_attr = edge_attr
    return data


for data in pyg_list:
    data = add_edge_features(data)


In [19]:
torch.save(pyg_list,'/home/runchangjia/project2/uncertain_list.pt')

In [20]:
import torch
pyg_list = torch.load('/home/runchangjia/project2/uncertain_list.pt')

In [ ]:
# load ESMC embedd and match with graph by uniprot id.
with open('/home/runchangjia/project2/esmc_win15_embedd/uncertain_window15_features.pkl', 'rb') as f:
    path_window15_features = pickle.load(f)



In [22]:
# 创建ID到embedd的映射字典
id2embedd = {}
for i, id_val in enumerate(path_window15_features['ID']):
    id2embedd[id_val] = path_window15_features['embedd'][i]

# 根据pyg_list中的id匹配并赋值embedd到.s属性
for k, v in enumerate(pyg_list):
    pyg_id = pyg_list[k].id
    if pyg_id in id2embedd:
        pyg_list[k].s = id2embedd[pyg_id]
    else:
        print(f"Warning: ID {pyg_id} not found in path_window15_features")
    

In [23]:
len(pyg_list)

611

In [25]:
pyg_list

[Data(edge_index=[2, 44], node_id=[14], coords=[14, 3], name='O00187', num_nodes=14, id='O00187_128', x=[14, 81], y=[1], edge_attr=[44, 7], s=[15, 2560]),
 Data(edge_index=[2, 46], node_id=[16], coords=[16, 3], name='O00217', num_nodes=16, id='O00217_102', x=[16, 81], y=[1], edge_attr=[46, 7], s=[15, 2560]),
 Data(edge_index=[2, 32], node_id=[11], coords=[11, 3], name='O00754', num_nodes=11, id='O00754_99', x=[11, 81], y=[1], edge_attr=[32, 7], s=[15, 2560]),
 Data(edge_index=[2, 38], node_id=[13], coords=[13, 3], name='O00754', num_nodes=13, id='O00754_1000', x=[13, 81], y=[1], edge_attr=[38, 7], s=[15, 2560]),
 Data(edge_index=[2, 36], node_id=[13], coords=[13, 3], name='O00754', num_nodes=13, id='O00754_956', x=[13, 81], y=[1], edge_attr=[36, 7], s=[15, 2560]),
 Data(edge_index=[2, 22], node_id=[8], coords=[8, 3], name='O00754', num_nodes=8, id='O00754_950', x=[8, 81], y=[1], edge_attr=[22, 7], s=[15, 2560]),
 Data(edge_index=[2, 42], node_id=[13], coords=[13, 3], name='O00754', num

In [26]:
torch.save(pyg_list,'/home/runchangjia/project2/uncertain_pyg_seq_list.pt')